#### ▶ Autograd(자동미분)
: PyTorch에서 연산과정을 추적하고 역전파를 통해 미분값을 자동으로 계산하는 기능
- **미분 계산 자동화** 
- **기울기 계산** : 각 입력값과 모델의 가중치가 손실함수에 미치는 영향 계산
- **모델 학습 지원** : 계산한 기울기를 Optimizer에 전달하여 가중치 업데이트

In [1]:
import torch

##### Autograd의 기본 흐름
1. 미분할 텐서에 requires_grad = True설정
2. 해당 텐서를 사용하여 연산 수행
3. 결과 텐서에 backward() 실행.
4. 원본 텐서의 grad속성에서 미분값 확인

#### 1. requires_grad
- True : 해당 텐서에 대한 연산을 추적하여 기울기를 계산할 수 있도록 설정.
- False : 기본값, 해당 텐서에 대한 연산을 추적하지 않음
- 일반적으로 모델이 학습해야 하는 가중치(weight)와 편향(bias)에 True를 설정함

In [2]:
x = torch.tensor(1.0, requires_grad=True)
print(x)

tensor(1., requires_grad=True)


In [5]:
x = torch.tensor(1.0)
print(x)
print(x.requires_grad)

x.requires_grad=True
print(x)
print(x.requires_grad)

tensor(1.)
False
tensor(1., requires_grad=True)
True


In [7]:
# 1.
x = torch.tensor([1.0], requires_grad=True )

# 2. 함수 계산
y = x**2
print(y)

# 3. 역전파 실행
y.backward()

# 4. 결과와 기울기 확인
print('y :', y)
print('x의 기울기 :', x.grad)

tensor([1.], grad_fn=<PowBackward0>)
y : tensor([1.], grad_fn=<PowBackward0>)
x의 기울기 : tensor([2.])


In [10]:
x = torch.tensor(1.0, requires_grad=True)
x.requires_grad=False
    # transfer leanrning할 때 사용
    # 내가 만들어 놓은 학습된 모델에서 일부부만 학습시키고 싶을 때 나머지 부분만
y = x**2
print(y)
# y.backward() # error

tensor(1.)


In [11]:
x = torch.tensor([2.0, 4.0], requires_grad=True)
y = x * 3

print('x.requires_grad :', x.requires_grad)
print('y.requires_grad :', y.requires_grad)
print('x.is_leaf :', x.is_leaf)
print('y.is_leaf :', y.is_leaf)
print('y.grad_fn :', y.grad_fn)

x.requires_grad : True
y.requires_grad : True
x.is_leaf : True
y.is_leaf : False
y.grad_fn : <MulBackward0 object at 0x000001FA716D9A80>


#### 2. 연쇄법칙(Chain Rule)
함수가 여러 연산으로 연결되어 있으면 각 연산의 미분값을 연쇄적으로 곱하여 최종 기울기를 계산함

In [14]:
x = torch.tensor(1.0, requires_grad=True)
y = x**2
print(y)

z = 3*y
print(z)

z.backward()
print(x.grad)
print(y.grad)



tensor(1., grad_fn=<PowBackward0>)
tensor(3., grad_fn=<MulBackward0>)
tensor(6.)
None


C:\Users\mega\AppData\Local\Temp\ipykernel_19372\582138082.py:10: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\build\aten\src\ATen/core/TensorBody.h:493.)
  print(y.grad)


In [15]:
# 중간에 있는 y의 미분값도 알고 싶다면
x = torch.tensor(1., requires_grad=True)
y = x**2
y.retain_grad()

z = 3*y

z.backward()
print('x의 미분값 :',x.grad)
print('y의 미분값 :',y.grad)

x의 미분값 : tensor(6.)
y의 미분값 : tensor(3.)


In [ ]:
x = torch.tensor(1., requires_grad=True)
y = x**2
z = 3*y

y.backward()    # y부터 미분 시작(z는 안함)
print(x.grad)

tensor(2.)


In [18]:
x = torch.tensor(1., requires_grad=True)
a = x**2
b = a+1
print(b)

c = b**2
c.backward()
print(x.grad)

tensor(2., grad_fn=<AddBackward0>)
tensor(8.)


#### 3. 편미분
: 함수에 여러 변수가 있을 때는 각 변수에 대해 따로 미분할 수 있음.

In [20]:
x = torch.tensor(1., requires_grad=True)
y = torch.tensor(1., requires_grad=True)

z = 2*x**2 + y**2
print(z)

z.backward()
print(x.grad)
print(y.grad)

tensor(3., grad_fn=<AddBackward0>)
tensor(4.)
tensor(2.)


In [22]:
x = torch.tensor(2., requires_grad=True)
y = torch.tensor(3., requires_grad=True)

f = x**2 + 2*x*y + y**2
f.backward()

print(x.grad)
print(y.grad)

tensor(10.)
tensor(10.)


#### 4. 벡터 출력의 기울기 계산
- 방법 1.

y.sum().backward()

- 방법 2.

y.backward(torch.ones_like(y))

In [26]:
x = torch.tensor([1., 2, 3], requires_grad=True)
y = torch.sum(x**2)

y.backward()
print('y :', y)
print('x.grad :', x.grad)
print('-' * 30)

x = torch.tensor([1., 2, 3], requires_grad=True)
y=x**2
y.sum().backward()

print('y :', y)
print('x.grad :', x.grad)
print('-' * 30)

x = torch.tensor([1., 2, 3], requires_grad=True)
y=x**2
y.backward(torch.ones_like(y))

print('y :', y)
print('x.grad :', x.grad)

y : tensor(14., grad_fn=<SumBackward0>)
x.grad : tensor([2., 4., 6.])
------------------------------
y : tensor([1., 4., 9.], grad_fn=<PowBackward0>)
x.grad : tensor([2., 4., 6.])
------------------------------
y : tensor([1., 4., 9.], grad_fn=<PowBackward0>)
x.grad : tensor([2., 4., 6.])


#### 5. Gradient 누적과 초기화
: backward()를 여러번 실행하면 grad에 새로운 기울기가 더해짐

- 기울기를 새로 계산하려면 기존 기울기를 0으로 초기화
    - x.grad.zero_()
    - 모델 학습에서는 일반적으로 optimizer.zero_grad()를 사용함

In [30]:
x = torch.tensor([1., 2, 3], requires_grad=True)

y = (x**2).sum()
y.backward()
print('첫번째 backward :', x.grad)

z = (3*x).sum()
z.backward()
print('두번째 backward 후 누적 :', x.grad)

x.grad.zero_()
print('초기화 후 :', x.grad)

w = (x**3).sum()
w.backward()
print('초기화 후 새로 계산 :', x.grad)

첫번째 backward : tensor([2., 4., 6.])
두번째 backward 후 누적 : tensor([5., 7., 9.])
초기화 후 : tensor([0., 0., 0.])
초기화 후 새로 계산 : tensor([ 3., 12., 27.])
